# PGD Attack — All Hyperspectral Datasets

Runs PGD on: **PaviaU · Salinas · Houston · Indian Pines**

Computes: **OA · Kappa · AA · SAM · SID · ASR · Physical-Consistency Rate**

Exports results to `PGD_All_Metrics_Results.xlsx` saved to your Google Drive.

In [ ]:
# ── Step 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Step 2: Clone / pull latest code from GitHub ─────────────────────────────
import os

REPO = '/content/S3ANET'
if not os.path.exists(REPO):
    !git clone https://github.com/SRUJANPATEL3669/S3ANET.git {REPO}
else:
    !git -C {REPO} pull

%cd {REPO}
!pip install -q openpyxl h5py

In [ ]:
# ── Step 3: Copy datasets from Drive → ./Data/ ────────────────────────────────
import shutil

DRIVE_DATA = '/content/drive/MyDrive/S3ANet_data'   # <-- your Drive folder name
LOCAL_DATA = './Data'
os.makedirs(LOCAL_DATA, exist_ok=True)

copied = []
for fname in os.listdir(DRIVE_DATA):
    if fname.endswith('.mat'):
        shutil.copy2(os.path.join(DRIVE_DATA, fname),
                     os.path.join(LOCAL_DATA,  fname))
        copied.append(fname)

print(f'Copied {len(copied)} files to {LOCAL_DATA}:')
print('  ', '\n  '.join(sorted(copied)))

In [ ]:
# ── Step 4: Generate train/test splits for every dataset ──────────────────────
for data_id in range(1, 5):
    print(f'\n--- GenSample dataID={data_id} ---')
    !python GenSample.py --dataID {data_id} --train_samples 200

In [ ]:
# ── Step 5: Run PGD on ALL datasets ─────────────────────────────────────────
#   --dataID 0  →  runs all 4 datasets in sequence
#   Adjust --epoch, --epsilon, --iters, and --restarts as needed
!python Attack_PGD_S3ANet.py --dataID 0 --epoch 1000 --epsilon 0.04 --iters 10 --restarts 5

In [ ]:
# ── Step 6: Preview and save results to Drive ─────────────────────────────────
import pandas as pd

results_file = 'PGD_All_Metrics_Results.xlsx'
df = pd.read_excel(results_file)
display(df)

dest = f'/content/drive/MyDrive/{results_file}'
shutil.copy2(results_file, dest)
print(f'\nSaved to Google Drive: {dest}')